# 05 · Especialização por agente (comparada ao RAG simples)

Escolhemos **agente** (e não LoRA) como especialização: roda em GPU de 8 GB sem treino, é auditável passo a passo e ataca diretamente o problema de alucinação. Um LoRA sobre poucas dezenas de trechos tenderia a *decorar* e não a recusar.

**Passos do agente (config C):**
1. **Reescrever** a pergunta em consulta de busca (LLM).
2. **Portão de relevância:** se a similaridade máxima < limiar calibrado (notebook 03) → recusa.
3. **Busca híbrida** (embeddings + BM25, RRF).
4. **Gerar** resposta fundamentada em JSON.
5. **Verificar:** (a) *determinístico*: citações existem e todo número da resposta está nos trechos citados; (b) *verificador LLM*: toda afirmação é sustentada?
6. Se falhar, **regenera uma vez** com o motivo; se falhar de novo, **recusa**.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)
from rotalivre_ai import config, kb, metrics, montar, pipelines
from rotalivre_ai.runner import checar_ollama, rodar_todas
print(checar_ollama())

Ollama OK em http://localhost:11434 | LLM=qwen2.5:7b | embeddings=bge-m3


In [2]:
trechos, be, idx = montar()
print("limiar em uso:", round(config.limiar(), 3))
def mostrar(q):
    r = pipelines.responder_agente(q, idx, be)
    print("PERGUNTA:", q); print("RESPOSTA:", r.resposta, "| fontes:", r.trecho_ids, "| recusou:", r.recusou)
    for p in r.trace: print("   ·", p)
    print()
    return r
for q in ["Qual a altura mínima de toldos e marquises sobre a calçada?",
          "Qual a inclinação longitudinal máxima permitida para a calçada?",      # armadilha N03
          "Qual a iluminância mínima, em lux, exigida para calçadas?"]:
    mostrar(q)

limiar em uso: 0.5
PERGUNTA: Qual a altura mínima de toldos e marquises sobre a calçada?
RESPOSTA: A altura mínima de toldos e marquises sobre a calçada é de 3,00 m. | fontes: ['T22'] | recusou: False
   · {'passo': 'reescrever', 'consulta': 'altura mínima toldos marquises calçada NBR 9050', 'tema': 'norma'}
   · {'passo': 'portao', 'cos_max': 0.781, 'limiar': 0.5}
   · {'passo': 'busca_hibrida', 'ids': ['T22', 'T17', 'T25']}
   · {'passo': 'gerar_1', 'json_valido': True, 'ids': ['T22']}
   · {'passo': 'verificar_1', 'deterministico': True, 'motivo': '', 'llm': {'suportada': True, 'problemas': ''}}

PERGUNTA: Qual a inclinação longitudinal máxima permitida para a calçada?
RESPOSTA: A inclinação máxima permitida para a rampa de rebaixamento da guia é de 8,33%. | fontes: ['T18'] | recusou: False
   · {'passo': 'reescrever', 'consulta': 'inclinação longitudinal máxima calçada NBR 9050', 'tema': 'norma'}
   · {'passo': 'portao', 'cos_max': 0.656, 'limiar': 0.5}
   · {'passo': 'busca_hibrid

In [3]:
# B x C nas perguntas-armadilha (não consta)
casos = [c for c in kb.carregar_casos() if c["tipo"] == "nao_consta"]
linhas = []
for c in casos:
    b = pipelines.responder_rag(c["pergunta"], idx, be); a = pipelines.responder_agente(c["pergunta"], idx, be)
    linhas.append({"id": c["id"], "pergunta": c["pergunta"], "B recusou": b.recusou, "C recusou": a.recusou, "B respondeu": "" if b.recusou else b.resposta})
r = pd.DataFrame(linhas)
print(f"recusa correta: B = {r['B recusou'].mean():.0%} | C = {r['C recusou'].mean():.0%}")
r

recusa correta: B = 100% | C = 88%


,id,pergunta,B recusou,C recusou,B respondeu
0,N01,Qual o valor em reais da multa por estacionar na calçada?,True,True,
1,N02,"Qual a iluminância mínima, em lux, exigida para calçadas?",True,True,
2,N03,Qual a inclinação longitudinal máxima permitida para a calçada?,True,False,
3,N04,Quantas pessoas com deficiência vivem no estado de São Paulo?,True,True,
4,N05,Qual a distância máxima permitida entre postes de iluminação pública?,True,True,
5,N06,Qual a largura mínima de uma ciclovia segundo a NBR 9050?,True,True,
6,N07,Qual o preço do sensor LDR usado no robô?,True,True,
7,N08,Qual a largura mínima de uma vaga de estacionamento para pessoa com deficiência?,True,True,


### Explicabilidade e pessoa no circuito
* Toda resposta traz **fontes** (id, citação, URL, licença) e o **trace** dos passos (campo `explicacao` da API).
* A API **não decide** nada pelo cidadão: o endpoint `/evento` explica o que o robô viu, cita a regra e recomenda **verificação humana**.
* O modelo não recebe dados pessoais (só pergunta e dados do sensor).